# FedKG-Maint — Phase 3 v2: Federated Learning + Privacy Trade-off (prep_v3)

This notebook trains **FedAvg**, **FedProx**, **DP-FedAvg**, and **SecureAggSim-FedAvg** using the four training factories in `prep_v3`. It includes matched-architecture centralized and per-client local-only MLP baselines, then compares them with Phase 2's selected centralized references. A validation-only DP noise sweep quantifies the client-level privacy/utility trade-off without using test scores.

- `SecureAggSim-FedAvg` is only a pairwise-mask arithmetic simulation. It is not cryptographic secure aggregation and must not be presented as a deployed privacy guarantee.

## Evaluation contract

- `factory_1`–`factory_4` are the only training clients. `factory_5` is held out from supervised training and validation-based selection.
- Hyperparameters and checkpoints are selected using `validation` only. Test sets are loaded for scoring only after all model choices are locked.
- Primary held-out-site evaluation: `test_site_full`. Secondary sets: `test_site_post`, `test_time_late`, and `test_site_late`. The two held-out-site subsets are nested within `test_site_full`, not independent replications.
- `global` and `local` normalization are evaluated separately. `local` uses Phase 1's feature-only, unlabeled calibration period for the held-out factory; this is unsupervised adaptation, not zero-shot transfer.
- DP-FedAvg clips each client's model update and adds Gaussian noise to the aggregated update. The notebook reports a client-level RDP bound for the **training-update channel under a trusted server**. The SecureAggSim method verifies pairwise-mask cancellation but is not a cryptographic secure-aggregation implementation and makes no standalone security claim. It does not claim end-to-end DP for centralized validation selection, published metrics, normalization statistics, or raw-data access by the evaluator.

This phase is the federated-learning baseline. SecureAggSim is only a protocol-arithmetic demonstration, not actual cryptographic protection. The maintenance knowledge graph and grounded LLM explanations remain later project components.

## 1. Setup and artifact checks

Run Phase 1 v3 and Phase 2 v3 first. This notebook writes to a new `phase3_federated_prep_v3_privacy_v2/` directory and does not overwrite prior outputs. The default is three random seeds; set `SEEDS = [42]` for a smoke test, not for final reported results.

In [ ]:
import os, sys, json, copy, time, math, random, warnings, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import average_precision_score

import torch
from torch import nn
from torch.nn.utils import parameters_to_vector, vector_to_parameters

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/FedKG-Maint")
except ImportError:
    PROJECT_DIR = Path(os.environ.get("FEDKG_ROOT", "./FedKG-Maint")).resolve()

PREP_DIR = PROJECT_DIR / "prep_v3"
P2_DIR = PROJECT_DIR / "phase2_baselines_prep_v3"
P3_DIR = PROJECT_DIR / "phase3_federated_prep_v3_privacy_v2"
for sub in ["models", "predictions", "results", "reports", "logs"]:
    (P3_DIR / sub).mkdir(parents=True, exist_ok=True)

CONFIG = {
    "SEEDS": [42, 43, 44],
    "MAX_ROUNDS": 12,
    "BATCH_SIZE": 4096,
    "LEARNING_RATE": 1e-3,
    "WEIGHT_DECAY": 1e-5,
    "POS_WEIGHT": 20.0,       # fixed a priori; not computed from client label counts
    "HIDDEN": [64, 32],
    "DROPOUT": 0.10,
    "FEDPROX_MU": 0.01,
    "DP_CLIP_NORM": 1.0,
    "DP_NOISE_MULTIPLIER": 4.0,
    "DP_DELTA": 1e-5,
    "BOOTSTRAP_B": 500,
    "CI_LEVEL": 0.95,
    "FEATURE_SETS": ["full", "no_maintenance"],
    "POLICIES": ["global", "local"],
    "CLIENTS": [f"factory_{i}" for i in range(1, 5)],
    "HELD_OUT": "factory_5",
    "PRIMARY_SET": "test_site_full",
    "EVAL_SETS": ["test_site_full", "test_site_post", "test_time_late", "test_site_late"],
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "SEED_BOOTSTRAP": 13031,
}
if os.environ.get("FEDKG_FAST"):
    CONFIG.update(SEEDS=[42], MAX_ROUNDS=3, BATCH_SIZE=8192, BOOTSTRAP_B=40)

missing = []
for p in [PREP_DIR / "fedkg_data.py", PREP_DIR / "configs" / "feature_groups.json",
          P2_DIR / "fedkg_eval.py", P2_DIR / "reports" / "phase2_baselines_report.json"]:
    if not p.exists(): missing.append(str(p))
if missing:
    raise FileNotFoundError("Run Phase 1 v3 and Phase 2 v3 first. Missing artifacts:\n" + "\n".join(missing))

sys.path.insert(0, str(PREP_DIR))
sys.path.insert(0, str(P2_DIR))
sys.modules.pop("fedkg_data", None)
sys.modules.pop("fedkg_eval", None)
importlib.invalidate_caches()
import fedkg_data
from fedkg_eval import SetEval, draw_weights, ci

FEATURE_CFG = json.loads((PREP_DIR / "configs" / "feature_groups.json").read_text())
TARGET = FEATURE_CFG["target"]
ALL_FEATURES = FEATURE_CFG["feature_sets"]["full"]
TRAIN_CUTOFF = pd.Timestamp(FEATURE_CFG["boundaries"]["train_cutoff"])
P2_REPORT = json.loads((P2_DIR / "reports" / "phase2_baselines_report.json").read_text())
DEVICE = torch.device(CONFIG["DEVICE"])

assert len(ALL_FEATURES) == 25
assert set(CONFIG["CLIENTS"]).isdisjoint({CONFIG["HELD_OUT"]})
assert set(fedkg_data.SPLITS) >= {"train", "validation", "test_time_late", "test_site_late", "test_site_full"}
print("Device:", DEVICE)
print("Project:", PROJECT_DIR)
print("Prep inputs:", PREP_DIR)
print("Phase 2 references:", P2_DIR)
print("Phase 3 output:", P3_DIR)
print("Seeds:", CONFIG["SEEDS"], "| rounds:", CONFIG["MAX_ROUNDS"], "| batch:", CONFIG["BATCH_SIZE"])
print("Clients:", CONFIG["CLIENTS"], "| held-out:", CONFIG["HELD_OUT"])

## 2. Model and federated-update utilities

All learned Phase 3 methods use the same MLP architecture and fixed positive-class loss weight. FedAvg, FedProx, and SecureAggSim-FedAvg perform one local epoch per round and use equal client weights; Phase 1 assigns 20 machines and the same number of hourly rows to each training client. DP-FedAvg clips each client update in model-parameter space before averaging and adds Gaussian noise to the average update.

SecureAggSim-FedAvg adds pairwise masks that cancel in the aggregate. It is a protocol-logic simulation only: masks and updates exist in one Python process, so this is not production cryptography or proof of server-side confidentiality. DP-FedAvg remains the mechanism with a formal, scoped RDP accounting bound.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def make_model(n_features, seed):
    # Seed each initialization deterministically; training shuffles use the global torch RNG.
    torch.manual_seed(int(seed))
    h1, h2 = CONFIG["HIDDEN"]
    return nn.Sequential(
        nn.Linear(n_features, h1), nn.ReLU(), nn.Dropout(CONFIG["DROPOUT"]),
        nn.Linear(h1, h2), nn.ReLU(), nn.Dropout(CONFIG["DROPOUT"]),
        nn.Linear(h2, 1)
    ).to(DEVICE)


def cpu_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def restore_model(state, n_features, seed):
    model = make_model(n_features, seed)
    model.load_state_dict(state)
    return model


def data_tensor(X, y=None):
    xt = torch.as_tensor(np.asarray(X, dtype=np.float32), dtype=torch.float32, device=DEVICE)
    if y is None: return xt
    yt = torch.as_tensor(np.asarray(y, dtype=np.float32), dtype=torch.float32, device=DEVICE).reshape(-1, 1)
    return xt, yt


def train_one_epoch(model, X_t, y_t, row_indices, optimizer, *, global_model=None, prox_mu=0.0):
    model.train()
    order = row_indices[torch.randperm(len(row_indices), device=DEVICE)]
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([CONFIG["POS_WEIGHT"]], device=DEVICE))
    total_loss, seen = 0.0, 0
    for start in range(0, len(order), CONFIG["BATCH_SIZE"]):
        idx = order[start:start + CONFIG["BATCH_SIZE"]]
        xb = X_t.index_select(0, idx)
        yb = y_t.index_select(0, idx)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        if global_model is not None and prox_mu > 0:
            prox = torch.zeros((), device=DEVICE)
            for p_local, p_global in zip(model.parameters(), global_model.parameters()):
                prox = prox + torch.sum((p_local - p_global.detach()) ** 2)
            loss = loss + 0.5 * prox_mu * prox
        loss.backward()
        optimizer.step()
        n = len(idx)
        total_loss += float(loss.detach().cpu()) * n
        seen += n
    return total_loss / max(seen, 1)


@torch.no_grad()
def predict_scores(model, X, batch_size=32768):
    model.eval()
    xt = torch.as_tensor(np.asarray(X, dtype=np.float32), dtype=torch.float32, device=DEVICE)
    outs = []
    for start in range(0, len(xt), batch_size):
        logits = model(xt[start:start + batch_size]).reshape(-1)
        outs.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(outs) if outs else np.empty(0, dtype=np.float64)


def pooled_val_scores(local_models, X_val, meta_val, seed):
    scores = np.full(len(meta_val), np.nan, dtype=np.float64)
    for client in CONFIG["CLIENTS"]:
        mask = meta_val["client_id"].to_numpy() == client
        if mask.any(): scores[mask] = predict_scores(local_models[client], X_val[mask])
    if not np.isfinite(scores).all():
        raise ValueError("Local-only validation predictions do not cover every validation row.")
    return scores


def update_vector(model):
    return parameters_to_vector([p.detach() for p in model.parameters()]).detach().clone()


def load_vector(model, vector):
    vector_to_parameters(vector, model.parameters())


def secure_aggregate_mean(client_updates, seed, round_index, mask_scale=1.0):
    """Simulate pairwise-mask aggregation; NOT a cryptographic secure-aggregation implementation.

    Each pair derives a cancelling mask (+m for one participant, -m for the other).
    The coordinator-facing aggregate is the mean of masked updates; masks cancel in sum.
    In this notebook the masks are created in one process, so the function demonstrates
    protocol arithmetic only and must not be described as production confidentiality.
    """
    if not client_updates:
        raise ValueError("secure aggregation requires at least one client update")
    masked = [u.detach().clone() for u in client_updates]
    device = masked[0].device
    generator = torch.Generator(device=device)
    generator.manual_seed(int(seed) * 1_000_003 + int(round_index) * 9_176 + 53)
    for i in range(len(masked)):
        for j in range(i + 1, len(masked)):
            mask = torch.randn(masked[i].shape, dtype=masked[i].dtype,
                               device=device, generator=generator) * float(mask_scale)
            masked[i].add_(mask)
            masked[j].sub_(mask)
    return torch.stack(masked, dim=0).sum(dim=0) / len(masked)


def secure_aggregation_smoke_test():
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(20261016)
    updates = [torch.randn(257, device=DEVICE, generator=gen) for _ in range(4)]
    direct = torch.stack(updates, dim=0).mean(dim=0)
    masked = secure_aggregate_mean(updates, seed=7, round_index=3, mask_scale=1.0)
    err = float(torch.max(torch.abs(direct - masked)).detach().cpu())
    if not torch.allclose(direct, masked, atol=2e-5, rtol=2e-5):
        raise AssertionError(f"pairwise masks failed to cancel; max_abs_error={err}")
    print(f"[PASS] SECAGG-SIM-1 pairwise masks cancel in aggregate (max absolute error={err:.2e}); simulation only, no cryptographic claim")

secure_aggregation_smoke_test()


def best_validation_record(se_eval, y_val, scores):
    ap = float(average_precision_score(y_val, scores))
    packed = se_eval.pack(scores)
    threshold = se_eval.best_f1_threshold(packed)
    return ap, threshold


def fit_one_epoch_optimizer(model):
    return torch.optim.Adam(model.parameters(), lr=CONFIG["LEARNING_RATE"], weight_decay=CONFIG["WEIGHT_DECAY"])


def privacy_epsilon(noise_multiplier, steps, delta):
    """Full-participation, client-level Gaussian RDP upper bound; no client subsampling."""
    if noise_multiplier <= 0: return float("inf"), None
    orders = np.array([1.25, 1.5, 2, 3, 4, 5, 8, 10, 16, 32, 64, 128, 256, 512], dtype=float)
    rdp = steps * orders / (2.0 * noise_multiplier ** 2)
    eps = rdp + math.log(1.0 / delta) / (orders - 1.0)
    j = int(np.argmin(eps))
    return float(eps[j]), float(orders[j])


def train_federated(X_train, y_train, meta_train, X_val, y_val, meta_val,
                    feature_set, policy, seed, algorithm, val_eval, client_ids=None, client_column="client_id"):
    """Train FedAvg, FedProx or DP-FedAvg; checkpoint selection uses validation AP only."""
    seed_everything(seed)
    n_features = X_train.shape[1]
    clients = list(client_ids) if client_ids is not None else list(CONFIG["CLIENTS"])
    if client_column not in meta_train.columns:
        raise ValueError(f"Missing federated client column: {client_column}")
    X_t, y_t = data_tensor(X_train, y_train)
    client_idx = {c: torch.as_tensor(np.flatnonzero(meta_train[client_column].to_numpy() == c),
                                     dtype=torch.long, device=DEVICE) for c in clients}
    sizes = {c: int(len(client_idx[c])) for c in clients}
    if any(n == 0 for n in sizes.values()): raise ValueError(f"Missing training client: {sizes}")
    if len(set(sizes.values())) != 1:
        raise ValueError(f"This implementation uses equal client weights and requires equal training sizes: {sizes}")

    global_model = make_model(n_features, seed)
    history, best = [], {"ap": -np.inf, "round": None, "state": None, "scores": None, "threshold": None}
    t0 = time.time()
    for rnd in range(1, CONFIG["MAX_ROUNDS"] + 1):
        base = update_vector(global_model)
        deltas = []
        for client in clients:
            local_model = make_model(n_features, seed + rnd * 100 + int(client.split("_")[-1]))
            local_model.load_state_dict(global_model.state_dict())
            opt = fit_one_epoch_optimizer(local_model)
            mu = CONFIG["FEDPROX_MU"] if algorithm == "FedProx" else 0.0
            train_one_epoch(local_model, X_t, y_t, client_idx[client], opt,
                            global_model=global_model if mu > 0 else None, prox_mu=mu)
            deltas.append(update_vector(local_model) - base)
            del local_model, opt

        if algorithm == "DP-FedAvg":
            clipped = []
            C = CONFIG["DP_CLIP_NORM"]
            for delta_vec in deltas:
                norm = torch.linalg.vector_norm(delta_vec)
                factor = torch.clamp(torch.tensor(C, device=DEVICE) / (norm + 1e-12), max=1.0)
                clipped.append(delta_vec * factor)
            mean_delta = torch.stack(clipped, dim=0).mean(dim=0)
            # Mean-update sensitivity C / N_clients under add/remove adjacency with a fixed public denominator.
            noise_std = CONFIG["DP_NOISE_MULTIPLIER"] * C / len(clients)
            mean_delta = mean_delta + torch.normal(0.0, noise_std, size=mean_delta.shape, device=DEVICE)
        elif algorithm == "SecureAggSim-FedAvg":
            # Pairwise masks cancel only after aggregation; this is an arithmetic simulation, not cryptography.
            mean_delta = secure_aggregate_mean(deltas, seed=seed, round_index=rnd)
        else:
            mean_delta = torch.stack(deltas, dim=0).mean(dim=0)
        load_vector(global_model, base + mean_delta)

        val_scores = predict_scores(global_model, X_val)
        val_ap, threshold = best_validation_record(val_eval, y_val, val_scores)
        history.append({"seed": seed, "feature_set": feature_set, "policy": policy,
                        "method": algorithm, "round": rnd, "validation_ap": val_ap})
        if val_ap > best["ap"]:
            best = {"ap": val_ap, "round": rnd, "state": cpu_state(global_model),
                    "scores": val_scores.copy(), "threshold": float(threshold)}
    elapsed = time.time() - t0
    epsilon, order = (privacy_epsilon(CONFIG["DP_NOISE_MULTIPLIER"], CONFIG["MAX_ROUNDS"], CONFIG["DP_DELTA"])
                      if algorithm == "DP-FedAvg" else (np.nan, np.nan))
    return {"method": algorithm, "feature_set": feature_set, "policy": policy, "seed": seed,
            "n_features": n_features, "selected_round": int(best["round"]), "val_ap": float(best["ap"]),
            "threshold": float(best["threshold"]), "state": best["state"],
            "validation_scores": best["scores"], "fit_seconds": elapsed,
            "dp_epsilon_train_update_channel": epsilon, "dp_delta": CONFIG["DP_DELTA"] if algorithm == "DP-FedAvg" else np.nan,
            "dp_noise_multiplier": CONFIG["DP_NOISE_MULTIPLIER"] if algorithm == "DP-FedAvg" else np.nan,
            "dp_rdp_order": order, "client_ids": clients, "client_count": len(clients),
            "validation_history": history}


def train_central_mlp(X_train, y_train, X_val, y_val, meta_val, feature_set, policy, seed, val_eval):
    seed_everything(seed)
    X_t, y_t = data_tensor(X_train, y_train)
    idx = torch.arange(len(X_train), dtype=torch.long, device=DEVICE)
    model = make_model(X_train.shape[1], seed)
    opt = fit_one_epoch_optimizer(model)
    history, best = [], {"ap": -np.inf, "round": None}
    t0 = time.time()
    for epoch in range(1, CONFIG["MAX_ROUNDS"] + 1):
        train_one_epoch(model, X_t, y_t, idx, opt)
        scores = predict_scores(model, X_val)
        ap, thr = best_validation_record(val_eval, y_val, scores)
        history.append({"seed": seed, "feature_set": feature_set, "policy": policy,
                        "method": "CentralMLP", "round": epoch, "validation_ap": ap})
        if ap > best["ap"]:
            best = {"ap": ap, "round": epoch, "state": cpu_state(model), "scores": scores.copy(), "threshold": float(thr)}
    return {"method": "CentralMLP", "feature_set": feature_set, "policy": policy, "seed": seed,
            "n_features": X_train.shape[1], "selected_round": int(best["round"]), "val_ap": float(best["ap"]),
            "threshold": float(best["threshold"]), "state": best["state"],
            "validation_scores": best["scores"], "fit_seconds": time.time() - t0,
            "dp_epsilon_train_update_channel": np.nan, "dp_delta": np.nan,
            "dp_noise_multiplier": np.nan, "dp_rdp_order": np.nan,
            "client_ids": [], "client_count": 0, "validation_history": history}


def train_local_only(X_train, y_train, meta_train, X_val, y_val, meta_val,
                     feature_set, policy, seed, val_eval):
    seed_everything(seed)
    n_features = X_train.shape[1]
    X_t, y_t = data_tensor(X_train, y_train)
    client_idx = {c: torch.as_tensor(np.flatnonzero(meta_train["client_id"].to_numpy() == c),
                                     dtype=torch.long, device=DEVICE) for c in CONFIG["CLIENTS"]}
    models = {c: make_model(n_features, seed + 1000 + i) for i, c in enumerate(CONFIG["CLIENTS"])}
    opts = {c: fit_one_epoch_optimizer(models[c]) for c in CONFIG["CLIENTS"]}
    history, best = [], {"ap": -np.inf, "round": None}
    t0 = time.time()
    for epoch in range(1, CONFIG["MAX_ROUNDS"] + 1):
        for c in CONFIG["CLIENTS"]:
            train_one_epoch(models[c], X_t, y_t, client_idx[c], opts[c])
        scores = pooled_val_scores(models, X_val, meta_val, seed)
        ap, thr = best_validation_record(val_eval, y_val, scores)
        history.append({"seed": seed, "feature_set": feature_set, "policy": policy,
                        "method": "LocalOnly", "round": epoch, "validation_ap": ap})
        if ap > best["ap"]:
            best = {"ap": ap, "round": epoch, "state": {c: cpu_state(m) for c, m in models.items()},
                    "scores": scores.copy(), "threshold": float(thr)}
    return {"method": "LocalOnly", "feature_set": feature_set, "policy": policy, "seed": seed,
            "n_features": n_features, "selected_round": int(best["round"]), "val_ap": float(best["ap"]),
            "threshold": float(best["threshold"]), "state": best["state"],
            "validation_scores": best["scores"], "fit_seconds": time.time() - t0,
            "dp_epsilon_train_update_channel": np.nan, "dp_delta": np.nan,
            "dp_noise_multiplier": np.nan, "dp_rdp_order": np.nan,
            "client_ids": list(CONFIG["CLIENTS"]), "client_count": len(CONFIG["CLIENTS"]),
            "validation_history": history}

## 3. Train and select using validation only

This cell intentionally loads only `train` and `validation`. No test split is opened in this stage. For every seed × feature set × normalization policy, it trains FedAvg, FedProx, DP-FedAvg, a matched centralized MLP, and four independent local-only models. The best round is selected by pooled validation PR-AUC. The local-only baseline cannot produce a held-out-factory prediction because it has no model trained for `factory_5`.

In [ ]:
TRAINED = []
HISTORY_ROWS = []
SELECTION_LOG = []
train_start = time.time()

# A second four-client partition balances machine-model composition while preserving the same 80 training machines.
primary_manifest_path = PREP_DIR / "manifests" / "factory_manifest.csv"
if not primary_manifest_path.exists():
    raise FileNotFoundError(f"Missing Phase 1 primary factory manifest: {primary_manifest_path}")
PRIMARY_MANIFEST = pd.read_csv(primary_manifest_path)
TRAIN_MANIFEST = PRIMARY_MANIFEST[PRIMARY_MANIFEST["client_id"].isin(CONFIG["CLIENTS"])].copy()
BALANCED_QUOTAS = {
    "balanced_1": {"model1": 4, "model2": 4, "model3": 6, "model4": 6},
    "balanced_2": {"model1": 4, "model2": 4, "model3": 7, "model4": 5},
    "balanced_3": {"model1": 4, "model2": 4, "model3": 7, "model4": 5},
    "balanced_4": {"model1": 3, "model2": 4, "model3": 7, "model4": 6},
}
balanced_parts = []
for j, model_name in enumerate(sorted(TRAIN_MANIFEST["model"].unique())):
    model_rows = TRAIN_MANIFEST[TRAIN_MANIFEST["model"] == model_name].sort_values("machineID").sample(
        frac=1, random_state=CONFIG["SEEDS"][0] + 700 + j).reset_index(drop=True)
    cursor = 0
    for balanced_client, quota in BALANCED_QUOTAS.items():
        n_take = quota.get(model_name, 0)
        part = model_rows.iloc[cursor:cursor + n_take].copy()
        part["fl_client"] = balanced_client
        balanced_parts.append(part)
        cursor += n_take
    if cursor != len(model_rows):
        raise ValueError(f"Balanced model quota did not exhaust {model_name}: {cursor}/{len(model_rows)}")
BALANCED_MANIFEST = pd.concat(balanced_parts, ignore_index=True)
if not BALANCED_MANIFEST["machineID"].is_unique or set(BALANCED_MANIFEST["machineID"]) != set(TRAIN_MANIFEST["machineID"]):
    raise AssertionError("Balanced partition must assign every training machine exactly once.")
if not (BALANCED_MANIFEST.groupby("fl_client").size() == 20).all():
    raise AssertionError("Balanced partition must contain 20 machines per client.")
(P3_DIR / "results").mkdir(parents=True, exist_ok=True)
BALANCED_MANIFEST[["machineID", "fl_client", "model", "age"]].to_csv(
    P3_DIR / "results" / "balanced_model_mix_manifest.csv", index=False)
print("Balanced four-client model composition:")
print(pd.crosstab(BALANCED_MANIFEST["fl_client"], BALANCED_MANIFEST["model"]).to_string())

for seed in CONFIG["SEEDS"]:
    for feature_set in CONFIG["FEATURE_SETS"]:
        for policy in CONFIG["POLICIES"]:
            print(f"\n=== seed={seed} | features={feature_set} | normalization={policy} ===")
            Xtr, ytr, mtr = fedkg_data.load_split("train", policy=policy, feature_set=feature_set)
            Xv, yv, mv = fedkg_data.load_split("validation", policy=policy, feature_set=feature_set)
            assert set(mtr["client_id"].unique()) == set(CONFIG["CLIENTS"]), mtr["client_id"].unique()
            assert CONFIG["HELD_OUT"] not in set(mtr["client_id"].unique())
            assert CONFIG["HELD_OUT"] not in set(mv["client_id"].unique())
            assert np.isfinite(Xtr).all() and np.isfinite(Xv).all()
            # Failure events are restricted to training-factory machines for validation event metrics.
            raw_failures_train = pd.read_csv(PROJECT_DIR / "data" / "raw" / "PdM_failures.csv", parse_dates=["datetime"])
            trainval_machines = set(mv["machineID"].unique())
            fe_trainval = (raw_failures_train[raw_failures_train["machineID"].isin(trainval_machines)]
                           [["machineID", "datetime"]].drop_duplicates().sort_values(["machineID", "datetime"]))
            val_eval = SetEval(mv.reset_index(drop=True), yv, fe_trainval)

            run_specs = [
                ("FedAvg", lambda: train_federated(Xtr, ytr, mtr, Xv, yv, mv, feature_set, policy, seed, "FedAvg", val_eval)),
                ("FedProx", lambda: train_federated(Xtr, ytr, mtr, Xv, yv, mv, feature_set, policy, seed, "FedProx", val_eval)),
                ("DP-FedAvg", lambda: train_federated(Xtr, ytr, mtr, Xv, yv, mv, feature_set, policy, seed, "DP-FedAvg", val_eval)),
                ("SecureAggSim-FedAvg", lambda: train_federated(Xtr, ytr, mtr, Xv, yv, mv, feature_set, policy, seed, "SecureAggSim-FedAvg", val_eval)),
                ("CentralMLP", lambda: train_central_mlp(Xtr, ytr, Xv, yv, mv, feature_set, policy, seed, val_eval)),
                ("LocalOnly", lambda: train_local_only(Xtr, ytr, mtr, Xv, yv, mv, feature_set, policy, seed, val_eval)),
            ]
            for method, fn in run_specs:
                artifact = fn()
                # Keep validation scores only for threshold/selection audit; model checkpoint is already locked.
                artifact_key = f"{method}|{feature_set}|{policy}|seed{seed}"
                artifact["key"] = artifact_key
                artifact["validation_history"] = artifact.get("validation_history", [])
                HISTORY_ROWS.extend(artifact["validation_history"])
                TRAINED.append(artifact)
                if method in ("FedAvg", "FedProx", "DP-FedAvg", "SecureAggSim-FedAvg"):
                    n_params = sum(v.numel() for v in artifact["state"].values())
                    one_model_bytes = n_params * 4  # float32 parameters; excludes transport/protocol overhead
                    n_clients = int(artifact.get("client_count", len(CONFIG["CLIENTS"])))
                    artifact["model_parameters"] = int(n_params)
                    artifact["model_payload_bytes"] = int(one_model_bytes)
                    artifact["estimated_train_communication_bytes"] = int(CONFIG["MAX_ROUNDS"] * n_clients * 2 * one_model_bytes)
                else:
                    artifact["model_parameters"] = sum(v.numel() for v in next(iter(artifact["state"].values())).values()) if method == "LocalOnly" else sum(v.numel() for v in artifact["state"].values())
                    artifact["model_payload_bytes"] = int(artifact["model_parameters"] * 4)
                    artifact["estimated_train_communication_bytes"] = np.nan
                SELECTION_LOG.append({"key": artifact_key, "method": method, "feature_set": feature_set,
                                      "policy": policy, "seed": seed, "selected_round": artifact["selected_round"],
                                      "validation_ap": artifact["val_ap"], "threshold_from_validation": artifact["threshold"],
                                      "selection_split": "validation"})
                print(f"  {method:12s} best round={artifact['selected_round']:2d} | validation AP={artifact['val_ap']:.4f} | {artifact['fit_seconds']:.1f}s")
                del fn
            # Sensitivity experiment: merge pairs of base factories into two equal-sized clients.
            # All 80 training machines and the same labels remain available; only client grouping changes.
            if feature_set == "full" and policy == "global":
                group_map = {"factory_1": "group_1", "factory_2": "group_1",
                             "factory_3": "group_2", "factory_4": "group_2"}
                mtr_group = mtr.copy(); mtr_group["fl_client"] = mtr_group["client_id"].map(group_map)
                for algorithm in ("FedAvg", "FedProx"):
                    art2 = train_federated(Xtr, ytr, mtr_group, Xv, yv, mv, feature_set, policy,
                                           seed, algorithm, val_eval, client_ids=["group_1", "group_2"],
                                           client_column="fl_client")
                    art2["method"] = f"{algorithm}_K2"
                    art2["key"] = f"{art2['method']}|{feature_set}|{policy}|seed{seed}"
                    art2["client_count_study"] = "2 grouped clients; same 80 training machines"
                    for history_row in art2.get("validation_history", []): history_row["method"] = art2["method"]
                    HISTORY_ROWS.extend(art2.get("validation_history", []))
                    n_params = sum(v.numel() for v in art2["state"].values())
                    payload = n_params * 4
                    art2["model_parameters"] = int(n_params)
                    art2["model_payload_bytes"] = int(payload)
                    art2["estimated_train_communication_bytes"] = int(CONFIG["MAX_ROUNDS"] * 2 * 2 * payload)
                    TRAINED.append(art2)
                    SELECTION_LOG.append({"key": art2["key"], "method": art2["method"],
                                          "feature_set": feature_set, "policy": policy, "seed": seed,
                                          "selected_round": art2["selected_round"], "validation_ap": art2["val_ap"],
                                          "threshold_from_validation": art2["threshold"], "selection_split": "validation"})
                    print(f"  {art2['method']:16s} best round={art2['selected_round']:2d} | validation AP={art2['val_ap']:.4f} | {art2['fit_seconds']:.1f}s")

                # Heterogeneity sensitivity: same 80 machines / four clients, but similar model-type mixes.
                mtr_bal = mtr.merge(BALANCED_MANIFEST[["machineID", "fl_client"]], on="machineID", how="left", validate="many_to_one")
                if mtr_bal["fl_client"].isna().any():
                    raise AssertionError("Balanced client map failed to cover training rows.")
                for algorithm in ("FedAvg", "FedProx"):
                    artb = train_federated(Xtr, ytr, mtr_bal, Xv, yv, mv, feature_set, policy,
                                           seed, algorithm, val_eval,
                                           client_ids=list(BALANCED_QUOTAS.keys()), client_column="fl_client")
                    artb["method"] = f"{algorithm}_balanced4"
                    artb["key"] = f"{artb['method']}|{feature_set}|{policy}|seed{seed}"
                    artb["partition_study"] = "balanced model mix; four clients; same 80 training machines"
                    for history_row in artb.get("validation_history", []): history_row["method"] = artb["method"]
                    HISTORY_ROWS.extend(artb.get("validation_history", []))
                    n_params_b = sum(v.numel() for v in artb["state"].values())
                    payload_b = n_params_b * 4
                    artb["model_parameters"] = int(n_params_b)
                    artb["model_payload_bytes"] = int(payload_b)
                    artb["estimated_train_communication_bytes"] = int(CONFIG["MAX_ROUNDS"] * 4 * 2 * payload_b)
                    TRAINED.append(artb)
                    SELECTION_LOG.append({"key": artb["key"], "method": artb["method"],
                                          "feature_set": feature_set, "policy": policy, "seed": seed,
                                          "selected_round": artb["selected_round"], "validation_ap": artb["val_ap"],
                                          "threshold_from_validation": artb["threshold"], "selection_split": "validation"})
                    print(f"  {artb['method']:16s} best round={artb['selected_round']:2d} | validation AP={artb['val_ap']:.4f} | {artb['fit_seconds']:.1f}s")

assert len(SELECTION_LOG) == len(CONFIG["SEEDS"]) * len(CONFIG["FEATURE_SETS"]) * len(CONFIG["POLICIES"]) * 6 + len(CONFIG["SEEDS"]) * 4
assert all(r["selection_split"] == "validation" for r in SELECTION_LOG)
print(f"\n[PASS] All model/checkpoint choices locked using validation only: {len(SELECTION_LOG)} artifacts")
print(f"Training stage wall time: {(time.time()-train_start)/60:.1f} minutes")

selection_df = pd.DataFrame(SELECTION_LOG)
selection_df.to_csv(P3_DIR / "results" / "validation_selection.csv", index=False)
pd.DataFrame(HISTORY_ROWS).to_csv(P3_DIR / "logs" / "validation_learning_curves.csv", index=False)
communication_rows = []
for art in TRAINED:
    communication_rows.append({"key": art["key"], "method": art["method"], "feature_set": art["feature_set"],
                               "policy": art["policy"], "seed": art["seed"],
                               "client_count": art.get("client_count", 0), "rounds_executed": CONFIG["MAX_ROUNDS"],
                               "model_parameters": art.get("model_parameters", np.nan),
                               "single_model_payload_bytes": art.get("model_payload_bytes", np.nan),
                               "estimated_train_communication_bytes": art.get("estimated_train_communication_bytes", np.nan),
                               "estimated_train_communication_MiB": (art.get("estimated_train_communication_bytes", np.nan) / (1024**2)) if np.isfinite(art.get("estimated_train_communication_bytes", np.nan)) else np.nan})
COMMUNICATION = pd.DataFrame(communication_rows)
COMMUNICATION.to_csv(P3_DIR / "results" / "communication_cost.csv", index=False)
for art in TRAINED:
    safe = art["key"].replace("|", "__")
    state_payload = {"state": art["state"], "method": art["method"], "feature_set": art["feature_set"],
                     "policy": art["policy"], "seed": art["seed"], "n_features": art["n_features"],
                     "selected_round": art["selected_round"], "validation_ap": art["val_ap"],
                     "threshold": art["threshold"], "config": CONFIG}
    torch.save(state_payload, P3_DIR / "models" / f"{safe}.pt")

In [ ]:
# Validation-only client-level DP utility/privacy sweep.
# The default DP-FedAvg setting (noise multiplier=4.0) remains predeclared in the main table.
# Lower multipliers are diagnostic only: they may improve utility but produce much weaker privacy.
_dp_sweep_seed = CONFIG["SEEDS"][0]
_dp_original_noise = float(CONFIG["DP_NOISE_MULTIPLIER"])
_dp_rows = []
try:
    _Xtr, _ytr, _mtr = fedkg_data.load_split("train", policy="global", feature_set="full")
    _Xv, _yv, _mv = fedkg_data.load_split("validation", policy="global", feature_set="full")
    _raw_f = pd.read_csv(PROJECT_DIR / "data" / "raw" / "PdM_failures.csv", parse_dates=["datetime"])
    _val_machines = set(_mv["machineID"].unique())
    _val_failures = (_raw_f[_raw_f["machineID"].isin(_val_machines)][["machineID", "datetime"]]
                     .drop_duplicates().sort_values(["machineID", "datetime"]))
    _val_eval = SetEval(_mv.reset_index(drop=True), _yv, _val_failures)
    # Reuse the already-computed default sigma=4 result for this seed/configuration.
    _base_dp = next(a for a in TRAINED if a["method"] == "DP-FedAvg" and a["feature_set"] == "full"
                    and a["policy"] == "global" and a["seed"] == _dp_sweep_seed)
    _noise_values = sorted(set([0.5, 1.0, 2.0, float(_dp_original_noise)]))
    for _sigma in _noise_values:
        if abs(_sigma - _dp_original_noise) < 1e-12:
            _a = _base_dp
        else:
            CONFIG["DP_NOISE_MULTIPLIER"] = float(_sigma)
            _a = train_federated(_Xtr, _ytr, _mtr, _Xv, _yv, _mv, "full", "global", _dp_sweep_seed,
                                 "DP-FedAvg", _val_eval)
        _eps, _order = privacy_epsilon(float(_sigma), CONFIG["MAX_ROUNDS"], CONFIG["DP_DELTA"])
        _nparams = sum(p.numel() for p in make_model(_Xtr.shape[1], _dp_sweep_seed).parameters())
        _ratio = float(_sigma * np.sqrt(_nparams) / len(CONFIG["CLIENTS"]))
        _dp_rows.append({"seed": _dp_sweep_seed, "feature_set": "full", "policy": "global",
                         "noise_multiplier": float(_sigma), "clip_norm": CONFIG["DP_CLIP_NORM"],
                         "epsilon_bound_training_update_channel": float(_eps), "delta": CONFIG["DP_DELTA"],
                         "selected_round_validation_only": int(_a["selected_round"]),
                         "validation_ap": float(_a["val_ap"]), "n_model_parameters": int(_nparams),
                         "expected_noise_l2_to_max_aggregate_signal_ratio_approx": _ratio,
                         "used_test_data": False})
finally:
    CONFIG["DP_NOISE_MULTIPLIER"] = _dp_original_noise
DP_SWEEP_TABLE = pd.DataFrame(_dp_rows).sort_values("noise_multiplier")
DP_SWEEP_TABLE.to_csv(P3_DIR / "results" / "dp_utility_privacy_sweep_validation_only.csv", index=False)
print("[PASS] DP sweep used validation only; no test split was loaded or scored by this cell.")
print(DP_SWEEP_TABLE.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("Interpretation: lower noise_multiplier can improve validation AP but raises epsilon; the signal/noise ratio is diagnostic, not a replacement for accounting.")


## 4. Lock audit and privacy-accounting notes

Before opening any test split, verify that every checkpoint has a validation-derived selection record. The reported epsilon applies to the client-update Gaussian mechanism with full participation and no subsampling amplification. It is a conservative training-update-channel bound; the notebook does **not** claim that a fully deployed FL system is end-to-end DP.

In [ ]:
assert len({a["key"] for a in TRAINED}) == len(TRAINED)
assert all(a["selected_round"] in range(1, CONFIG["MAX_ROUNDS"] + 1) for a in TRAINED)
assert all(np.isfinite(a["val_ap"]) and np.isfinite(a["threshold"]) for a in TRAINED)
assert not any(a["method"] == "LocalOnly" and a["state"] is None for a in TRAINED)
eps, rdp_order = privacy_epsilon(CONFIG["DP_NOISE_MULTIPLIER"], CONFIG["MAX_ROUNDS"], CONFIG["DP_DELTA"])
print("[PASS] Lock audit: test splits have not been loaded or scored in the selection stage.")
print(f"DP-FedAvg training-update accountant: noise multiplier={CONFIG['DP_NOISE_MULTIPLIER']}, rounds={CONFIG['MAX_ROUNDS']}, delta={CONFIG['DP_DELTA']}, epsilon_bound={eps:.3f}, selected_RDP_order={rdp_order}")
print("Scope caveat: this accountant excludes centralized access to validation labels/metrics, preprocessing statistics, and any external release of per-client diagnostics.")

## 5. Final evaluation — test sets are opened only after selection is locked

This stage predicts on the test splits with the checkpoints selected above. `test_site_post` is derived from `test_site_full` by retaining timestamps at/after the training cutoff; it is a nested subset. The local-only model is not evaluated on factory 5, because no local model was trained for that client.

In [ ]:
# Re-load and use all raw failure events only after every model/configuration is locked.
raw_failures_all = pd.read_csv(PROJECT_DIR / "data" / "raw" / "PdM_failures.csv", parse_dates=["datetime"])
FAIL_EVENTS = (raw_failures_all[["machineID", "datetime"]].drop_duplicates()
               .sort_values(["machineID", "datetime"]).reset_index(drop=True))

EVAL_CACHE = {}
for feature_set in CONFIG["FEATURE_SETS"]:
    for policy in CONFIG["POLICIES"]:
        for split in ["test_site_full", "test_time_late", "test_site_late"]:
            X, y, meta = fedkg_data.load_split(split, policy=policy, feature_set=feature_set)
            EVAL_CACHE[(feature_set, policy, split)] = (X, y, meta)
        # Phase 2 creates test_site_post as a time-filtered subset of test_site_full.
        Xf, yf, mf = EVAL_CACHE[(feature_set, policy, "test_site_full")]
        keep = mf["datetime"].to_numpy() >= np.datetime64(TRAIN_CUTOFF)
        EVAL_CACHE[(feature_set, policy, "test_site_post")] = (Xf[keep], yf[keep], mf.loc[keep].reset_index(drop=True))

# Verify the expected held-out/test boundaries after loading.
for feature_set in CONFIG["FEATURE_SETS"]:
    for policy in CONFIG["POLICIES"]:
        _, _, mfull = EVAL_CACHE[(feature_set, policy, "test_site_full")]
        _, _, mlate = EVAL_CACHE[(feature_set, policy, "test_site_late")]
        _, _, mtime = EVAL_CACHE[(feature_set, policy, "test_time_late")]
        assert set(mfull["client_id"].unique()) == {CONFIG["HELD_OUT"]}
        assert set(mlate["client_id"].unique()) == {CONFIG["HELD_OUT"]}
        assert set(mtime["client_id"].unique()) == set(CONFIG["CLIENTS"])
print("[PASS] Held-out test sets contain factory_5 only; temporal test contains factories 1–4.")
print("[PASS] Test predictions are being computed only after validation selection was locked.")

# Quantify observed client heterogeneity and save the available type-shift stress manifest.
Xhet, yhet, mhet = fedkg_data.load_split("train", policy="global", feature_set="full")
client_het = mhet[["machineID", "client_id"]].copy()
client_het["label"] = yhet
client_summary = client_het.groupby("client_id").agg(
    machines=("machineID", "nunique"), rows=("label", "size"), positive_rows=("label", "sum"),
    positive_rate=("label", "mean")).reset_index()
manifest_path = PREP_DIR / "manifests" / "factory_manifest.csv"
stress_manifest_path = PREP_DIR / "manifests" / "factory_manifest_type_shift_stress.csv"
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    mix = pd.crosstab(manifest["client_id"], manifest["model"], normalize="index").reset_index()
    mix.to_csv(P3_DIR / "results" / "client_model_mix.csv", index=False)
if stress_manifest_path.exists():
    stress_manifest = pd.read_csv(stress_manifest_path)
    stress_mix = pd.crosstab(stress_manifest["client_id"], stress_manifest["model"], normalize="index").reset_index()
    stress_mix.to_csv(P3_DIR / "results" / "type_shift_stress_manifest_mix.csv", index=False)
client_summary.to_csv(P3_DIR / "results" / "client_label_heterogeneity.csv", index=False)
print("Training-client label prevalence heterogeneity:")
print(client_summary.to_string(index=False))
print("[INFO] A four-client balanced-model partition is evaluated against the original non-IID partition; the extreme model-type-held-out manifest is summarized only, not used as the primary evaluation split.")

RESULT_ROWS = []
PREDICTIONS = {}
BOOT_REPS = {}
POINTS = {}

for art in TRAINED:
    method, fs, policy = art["method"], art["feature_set"], art["policy"]
    key = art["key"]
    n_features = art["n_features"]
    seed = art["seed"]
    # Recreate checkpoint(s) from CPU state dicts.
    if method == "LocalOnly":
        models = {c: restore_model(art["state"][c], n_features, seed + i)
                  for i, c in enumerate(CONFIG["CLIENTS"])}
    else:
        model = restore_model(art["state"], n_features, seed)

    for split in CONFIG["EVAL_SETS"]:
        Xev, yev, mev = EVAL_CACHE[(fs, policy, split)]
        # A local-only system has no model for an unseen factory: record as not applicable.
        if method == "LocalOnly" and set(mev["client_id"].unique()) - set(CONFIG["CLIENTS"]):
            RESULT_ROWS.append({"key": key, "method": method, "feature_set": fs, "policy": policy,
                                "seed": seed, "eval_set": split, "status": "not_applicable_no_heldout_model",
                                "ap": np.nan, "ap_ci_low": np.nan, "ap_ci_high": np.nan,
                                "roc_auc": np.nan, "precision": np.nan, "recall": np.nan, "f1": np.nan,
                                "fpr": np.nan, "event_recall": np.nan, "threshold": art["threshold"]})
            continue
        if method == "LocalOnly":
            score = pooled_val_scores(models, Xev, mev, seed)
        else:
            score = predict_scores(model, Xev)
        assert np.isfinite(score).all()
        PREDICTIONS[(key, split)] = score.astype(np.float32)
        evaluator = SetEval(mev.reset_index(drop=True), yev, FAIL_EVENTS)
        packed = evaluator.pack(score)
        point = evaluator.metrics(packed, None, art["threshold"])
        POINTS[(key, split)] = point
        ci_low, ci_high = np.nan, np.nan
        # Cluster bootstrap is concentrated on primary test and a temporal robustness set to control runtime.
        if split in (CONFIG["PRIMARY_SET"], "test_time_late"):
            boot_seed = CONFIG["SEED_BOOTSTRAP"] + (0 if split == CONFIG["PRIMARY_SET"] else 5000)
            W = draw_weights(evaluator.M, CONFIG["BOOTSTRAP_B"], boot_seed)
            reps = evaluator.bootstrap(packed, art["threshold"], W)
            BOOT_REPS[(key, split)] = reps
            ci_low, ci_high = ci(reps["ap"].dropna().to_numpy(), CONFIG["CI_LEVEL"])
        RESULT_ROWS.append({"key": key, "method": method, "feature_set": fs, "policy": policy,
                            "seed": seed, "eval_set": split, "status": "scored",
                            "ap": point["ap"], "ap_ci_low": ci_low, "ap_ci_high": ci_high,
                            "roc_auc": point["roc_auc"], "precision": point["precision"],
                            "recall": point["recall"], "f1": point["f1"], "fpr": point["fpr"],
                            "event_recall": point["event_recall"], "threshold": art["threshold"],
                            "n_machines": evaluator.M, "n_failure_events": evaluator.n_events,
                            "dp_epsilon_train_update_channel": art["dp_epsilon_train_update_channel"],
                            "dp_delta": art["dp_delta"], "dp_noise_multiplier": art["dp_noise_multiplier"]})

RESULTS = pd.DataFrame(RESULT_ROWS)
RESULTS.to_csv(P3_DIR / "results" / "federated_results.csv", index=False)
print("Saved scored results:", P3_DIR / "results" / "federated_results.csv")

## 6. Primary results and paired comparisons

The primary table uses `test_site_full`. Confidence intervals are machine-cluster bootstrap intervals; they quantify variation across the observed machines, not across independently sampled factories. Three seeds are reported separately and then summarized by method/configuration. Held-out test sets and any derived subsets are not used to select settings.

In [ ]:
PRIMARY = RESULTS[(RESULTS["eval_set"] == CONFIG["PRIMARY_SET"]) & (RESULTS["status"] == "scored")].copy()
cols = ["method", "feature_set", "policy", "seed", "ap", "ap_ci_low", "ap_ci_high", "roc_auc",
        "precision", "recall", "f1", "fpr", "event_recall", "dp_epsilon_train_update_channel"]
print(PRIMARY[cols].sort_values(["feature_set", "policy", "seed", "ap"], ascending=[True, True, True, False]).to_string(index=False))

SUMMARY = (PRIMARY.groupby(["method", "feature_set", "policy"], dropna=False)
           .agg(seed_count=("seed", "nunique"), mean_ap=("ap", "mean"), sd_ap=("ap", "std"),
                mean_roc_auc=("roc_auc", "mean"), mean_event_recall=("event_recall", "mean"),
                epsilon=("dp_epsilon_train_update_channel", "mean"))
           .reset_index())
SUMMARY.to_csv(P3_DIR / "results" / "primary_summary_by_seed.csv", index=False)

# Matched paired machine-bootstrap AP differences: FedAvg vs FedProx vs DP-FedAvg vs CentralMLP.
# Shared bootstrap weights are used by using a fixed seed for every model on the same primary set.
PAIR_ROWS = []
for fs in CONFIG["FEATURE_SETS"]:
    for policy in CONFIG["POLICIES"]:
        for seed in CONFIG["SEEDS"]:
            methods = ["FedAvg", "FedProx", "DP-FedAvg", "SecureAggSim-FedAvg", "CentralMLP"]
            lookup = {a["method"]: a for a in TRAINED if a["feature_set"] == fs and a["policy"] == policy and a["seed"] == seed}
            for i, a_method in enumerate(methods):
                for b_method in methods[i+1:]:
                    a = lookup[a_method]; b = lookup[b_method]
                    ka, kb = a["key"], b["key"]
                    ra = BOOT_REPS.get((ka, CONFIG["PRIMARY_SET"]))
                    rb = BOOT_REPS.get((kb, CONFIG["PRIMARY_SET"]))
                    if ra is None or rb is None: continue
                    delta = ra["ap"].to_numpy() - rb["ap"].to_numpy()
                    lo, hi = ci(delta, CONFIG["CI_LEVEL"])
                    POINT_DELTA = POINTS[(ka, CONFIG["PRIMARY_SET"])]["ap"] - POINTS[(kb, CONFIG["PRIMARY_SET"])]["ap"]
                    PAIR_ROWS.append({"feature_set": fs, "policy": policy, "seed": seed,
                                      "eval_set": CONFIG["PRIMARY_SET"], "A": a_method, "B": b_method,
                                      "delta_ap_A_minus_B": POINT_DELTA, "ci_low": lo, "ci_high": hi,
                                      "ci_excludes_zero": bool(lo > 0 or hi < 0)})
PAIRED = pd.DataFrame(PAIR_ROWS)
PAIRED.to_csv(P3_DIR / "results" / "paired_method_comparisons.csv", index=False)

PARTITION_ROWS = []
for seed in CONFIG["SEEDS"]:
    for algo in ("FedAvg", "FedProx"):
        variants = [algo, f"{algo}_K2", f"{algo}_balanced4"]
        for a_method, b_method in [(variants[0], variants[1]), (variants[0], variants[2]), (variants[1], variants[2])]:
            ka = f"{a_method}|full|global|seed{seed}"
            kb = f"{b_method}|full|global|seed{seed}"
            ra = BOOT_REPS.get((ka, CONFIG["PRIMARY_SET"]))
            rb = BOOT_REPS.get((kb, CONFIG["PRIMARY_SET"]))
            if ra is None or rb is None: continue
            drep = ra["ap"].to_numpy() - rb["ap"].to_numpy()
            lo, hi = ci(drep, CONFIG["CI_LEVEL"])
            PARTITION_ROWS.append({"seed": seed, "algorithm_family": algo, "feature_set": "full", "policy": "global",
                                   "eval_set": CONFIG["PRIMARY_SET"], "A": a_method, "B": b_method,
                                   "delta_ap_A_minus_B": POINTS[(ka, CONFIG["PRIMARY_SET"])]["ap"] - POINTS[(kb, CONFIG["PRIMARY_SET"])]["ap"],
                                   "ci_low": lo, "ci_high": hi, "ci_excludes_zero": bool(lo > 0 or hi < 0)})
PARTITION_CMP = pd.DataFrame(PARTITION_ROWS)
PARTITION_CMP.to_csv(P3_DIR / "results" / "partition_sensitivity_comparisons.csv", index=False)

print("\nSummary over seeds:")
print(SUMMARY.to_string(index=False))
print("\nPaired comparisons (primary test; bootstrap intervals):")
print(PAIRED.to_string(index=False) if len(PAIRED) else "No paired comparisons generated.")

## 7. Phase 2 centralized reference comparison

For each feature set and normalization policy, Phase 2 already selected a centralized reference on validation only. This table places its saved primary-set PR-AUC beside Phase 3 results. It does not retrain Phase 2 models, and it does not treat the HGB reference as an architecture-matched neural baseline; `CentralMLP` is the matched-architecture comparison for FL.

In [ ]:
REF_ROWS = []
ref_map = P2_REPORT.get("centralized_reference_per_feature_set_and_policy", {})
ref_head = P2_REPORT.get("headline_primary_set", {})
for combo, ref_key in ref_map.items():
    item = ref_head.get(ref_key, {})
    REF_ROWS.append({"feature_set_policy": combo, "phase2_reference_model": ref_key,
                     "phase2_test_site_full_ap": item.get("ap", np.nan),
                     "phase2_ci_low": (item.get("ci") or [np.nan, np.nan])[0],
                     "phase2_ci_high": (item.get("ci") or [np.nan, np.nan])[1]})
REF_TABLE = pd.DataFrame(REF_ROWS)
print(REF_TABLE.to_string(index=False))
REF_TABLE.to_csv(P3_DIR / "results" / "phase2_reference_comparison.csv", index=False)

## 8. Integrity checks, privacy report, and reproducible hand-off

Read the generated report before using these numbers in the project write-up. In particular, compare DP-FedAvg to FedAvg on the same seed / feature set / normalization policy, and keep the privacy scope caveat with any epsilon value.

In [ ]:
checks = {}
def check(name, ok, detail=""):
    checks[name] = {"status": "PASS" if bool(ok) else "FAIL", "detail": str(detail)}
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {detail}")
    if not ok: raise AssertionError(f"{name}: {detail}")

check("P3-FL-1 all client training data exclude held-out factory",
      set(fedkg_data.load_split("train", policy="global", feature_set="full")[2]["client_id"].unique()) == set(CONFIG["CLIENTS"]))
check("P3-FL-2 every model configuration selected on validation only",
      selection_df["selection_split"].eq("validation").all(), f"{len(selection_df)} selected checkpoints")
check("P3-FL-3 primary results use only post-lock scoring artifacts",
      set(RESULTS["eval_set"].dropna().unique()).issubset(set(CONFIG["EVAL_SETS"])))
check("P3-FL-4 local-only models are not scored on held-out factory",
      RESULTS[(RESULTS["method"] == "LocalOnly") & (RESULTS["eval_set"].isin(["test_site_full", "test_site_post", "test_site_late"]))]["status"].eq("not_applicable_no_heldout_model").all())
check("P3-FL-5 DP-FedAvg records positive finite privacy accountant bound",
      np.isfinite(privacy_epsilon(CONFIG["DP_NOISE_MULTIPLIER"], CONFIG["MAX_ROUNDS"], CONFIG["DP_DELTA"])[0])
      and privacy_epsilon(CONFIG["DP_NOISE_MULTIPLIER"], CONFIG["MAX_ROUNDS"], CONFIG["DP_DELTA"])[0] > 0)
check("P3-FL-6 every primary scored row has a finite PR-AUC",
      np.isfinite(PRIMARY["ap"].to_numpy()).all(), f"{len(PRIMARY)} primary rows")
check("P3-FL-7 SecureAggSim-FedAvg is present as a non-cryptographic simulation",
      any(a["method"] == "SecureAggSim-FedAvg" for a in TRAINED),
      "pairwise-mask cancellation is simulated; no cryptographic privacy claim")
check("P3-FL-8 validation-only DP sweep does not include test rows",
      (not DP_SWEEP_TABLE.empty) and DP_SWEEP_TABLE["used_test_data"].eq(False).all(),
      f"{len(DP_SWEEP_TABLE)} noise settings, validation only")

report = {
    "phase": "Phase 3 v2 Federated Learning + Privacy Trade-off",
    "input_preparation": "prep_v3",
    "phase2_reference_report": str(P2_DIR / "reports" / "phase2_baselines_report.json"),
    "output_directory": str(P3_DIR),
    "config": CONFIG,
    "methods": ["FedAvg", "FedProx", "DP-FedAvg", "SecureAggSim-FedAvg", "CentralMLP", "LocalOnly", "FedAvg_K2", "FedProx_K2", "FedAvg_balanced4", "FedProx_balanced4"],
    "client_count_sensitivity": "FedAvg/FedProx with four original training clients vs two merged clients (same 80 training machines), full features/global scaling",
    "heterogeneity_sensitivity": "FedAvg/FedProx on the same 80 training machines with the original mixed-model non-IID partition vs a balanced four-client model composition, full features/global scaling",
    "communication_cost": "estimated float32 model payload bytes: per round, each federated client downloads the global model and uploads one update; excludes protocol/security overhead",
    "selection_protocol": "best checkpoint by pooled validation PR-AUC only; threshold chosen on validation only",
    "primary_test_set": CONFIG["PRIMARY_SET"],
    "secondary_test_sets": CONFIG["EVAL_SETS"][1:],
    "held_out_factory": CONFIG["HELD_OUT"],
    "client_training_factories": CONFIG["CLIENTS"],
    "dp_utility_privacy_sweep_validation_only": DP_SWEEP_TABLE.to_dict("records"),
    "secure_aggregation_simulation": {
        "method": "pairwise masks added with opposite signs and cancelled in the aggregate",
        "status": "protocol-arithmetic simulation only; not cryptographic secure aggregation",
        "security_claim": "none; real confidentiality requires a vetted cryptographic protocol and threat-model testing"
    },
    "privacy": {
        "mechanism": "client-update l2 clipping plus Gaussian noise on averaged updates",
        "threat_model": "trusted server; privacy accounting applies to the training-update channel only",
        "adjacency": "client-level add/remove with fixed public client denominator",
        "sampling_amplification": "none; all four clients participate each round",
        "noise_multiplier": CONFIG["DP_NOISE_MULTIPLIER"],
        "clip_norm": CONFIG["DP_CLIP_NORM"],
        "rounds_accounted": CONFIG["MAX_ROUNDS"],
        "delta": CONFIG["DP_DELTA"],
        "epsilon_upper_bound": privacy_epsilon(CONFIG["DP_NOISE_MULTIPLIER"], CONFIG["MAX_ROUNDS"], CONFIG["DP_DELTA"])[0],
        "rdp_order": privacy_epsilon(CONFIG["DP_NOISE_MULTIPLIER"], CONFIG["MAX_ROUNDS"], CONFIG["DP_DELTA"])[1],
        "not_covered": ["centralized validation labels/metrics and checkpoint selection", "normalization statistics", "raw data access by a trusted server/evaluator", "external release of diagnostics"]
    },
    "phase2_centralized_references": ref_map,
    "checks": checks,
    "limitations": [
        "Only four simulated clients are available; privacy bounds with four clients should be interpreted cautiously.",
        "All clients participate every round; no subsampling amplification is claimed.",
        "DP accounting covers the clipped/noised training update mechanism, not validation selection or preprocessing statistics.",
        "Local normalization on factory_5 uses an earlier feature-only calibration interval; this is unsupervised adaptation.",
        "LocalOnly has no model for the held-out factory and is not applicable to held-out-site test rows.",
        "test_site_post and test_site_late are subsets of test_site_full, not independent replications.",
        "Results use simulated factories from one dataset, not independent real factories.",
        "Actual cryptographic secure aggregation is not implemented; SecureAggSim only demonstrates pairwise-mask cancellation in one process.",
        "The DP-FedAvg noise sweep is diagnostic; lower noise weakens the reported DP bound, and no test set is used to choose noise settings.",
        "Knowledge graph, LLM explanations and formal end-to-end privacy are not implemented in this phase."
    ]
}
(P3_DIR / "reports" / "phase3_federated_report.json").write_text(json.dumps(report, indent=2, default=str))

# Readable Markdown report with per-configuration references and primary outcomes.
md_lines = [
    "# FedKG-Maint Phase 3 v2 — Federated Learning + Privacy Trade-off", "",
    f"- Primary held-out factory set: `{CONFIG['PRIMARY_SET']}`",
    f"- Training clients: `{', '.join(CONFIG['CLIENTS'])}`; held-out: `{CONFIG['HELD_OUT']}`",
    f"- Seeds: `{CONFIG['SEEDS']}`; maximum rounds: `{CONFIG['MAX_ROUNDS']}`",
    f"- DP-FedAvg training-update epsilon bound: `{report['privacy']['epsilon_upper_bound']:.3f}` at delta `{CONFIG['DP_DELTA']}`",
    "- DP scope caveat: training-update channel under a trusted server only; validation selection and normalization statistics are excluded.",
    "- SecureAggSim-FedAvg only simulates cancelling pairwise masks in one process; it is not a cryptographic security implementation.", "",
    "## DP utility/privacy sweep (validation only)", "",
    "| Noise multiplier | Epsilon bound | Validation PR-AUC | Approx. noise/update norm ratio |", "|---:|---:|---:|---:|",
] + [f"| {r['noise_multiplier']:.2f} | {r['epsilon_bound_training_update_channel']:.3f} | {r['validation_ap']:.4f} | {r['expected_noise_l2_to_max_aggregate_signal_ratio_approx']:.2f} |" for _, r in DP_SWEEP_TABLE.iterrows()] + ["", "The lower-noise variants are diagnostic only; test results are not used to choose DP settings.", "",
    "## Primary results", "",
    "| Method | Features | Normalization | Seed | PR-AUC | 95% CI | ROC-AUC | Event recall | Epsilon bound |",
    "|---|---|---|---:|---:|---:|---:|---:|---:|"
]
for _, row in PRIMARY.sort_values(["feature_set", "policy", "seed", "ap"], ascending=[True, True, True, False]).iterrows():
    ci_text = f"{row['ap_ci_low']:.3f}–{row['ap_ci_high']:.3f}" if np.isfinite(row["ap_ci_low"]) else "—"
    er_text = f"{row['event_recall']:.3f}" if np.isfinite(row["event_recall"]) else "—"
    eps_text = f"{row['dp_epsilon_train_update_channel']:.3f}" if np.isfinite(row["dp_epsilon_train_update_channel"]) else "—"
    md_lines.append(f"| {row['method']} | {row['feature_set']} | {row['policy']} | {int(row['seed'])} | {row['ap']:.3f} | {ci_text} | {row['roc_auc']:.3f} | {er_text} | {eps_text} |")
ref_md = ["| Feature set / policy | Phase 2 reference model | AP | 95% CI |", "|---|---|---:|---:|"]
for _, rr in REF_TABLE.iterrows():
    ap_txt = f"{rr['phase2_test_site_full_ap']:.3f}" if np.isfinite(rr['phase2_test_site_full_ap']) else "N/A"
    lo_txt = f"{rr['phase2_ci_low']:.3f}" if np.isfinite(rr['phase2_ci_low']) else "N/A"
    hi_txt = f"{rr['phase2_ci_high']:.3f}" if np.isfinite(rr['phase2_ci_high']) else "N/A"
    ref_md.append(f"| {rr['feature_set_policy']} | {rr['phase2_reference_model']} | {ap_txt} | {lo_txt}–{hi_txt} |")
md_lines += ["", "## Phase 2 reference models", ""] + ref_md + ["", "## Estimated communication cost", ""]
for _, cc in COMMUNICATION.iterrows():
    value = f"{cc['estimated_train_communication_MiB']:.2f} MiB" if np.isfinite(cc['estimated_train_communication_MiB']) else "N/A"
    md_lines.append(f"- `{cc['key']}`: {value}")
md_lines += ["", "## Integrity checks", ""]
for name, result in checks.items(): md_lines.append(f"- **{result['status']}** — {name}: {result['detail']}")
md_lines += ["", "## Limitations", ""] + [f"- {x}" for x in report["limitations"]]
(P3_DIR / "reports" / "phase3_federated_report.md").write_text("\n".join(md_lines))

print("\nReports saved:", P3_DIR / "reports" / "phase3_federated_report.md")
print("JSON report:", P3_DIR / "reports" / "phase3_federated_report.json")
print("Results:", P3_DIR / "results" / "federated_results.csv")
print("PHASE 3 COMPLETE — review the privacy scope and held-out results before interpretation.")

## Interpretation rules

1. Compare FedAvg and FedProx with `CentralMLP` on the same feature set, normalization policy and seed; compare with the Phase 2 HGB reference as a separate, non-matched-architecture benchmark.
2. Compare DP-FedAvg against FedAvg under the same seed / feature / normalization setup, report PR-AUC alongside the scoped training-update epsilon bound, and include the validation-only noise sweep. Do not present SecureAggSim as production security.
3. Never describe this as end-to-end differential privacy. Validation-based model selection and preprocessing statistics are not included in the accountant.
4. Report `full` and `no_maintenance` results together. Phase 2 indicates maintenance recency is highly informative but may proxy time since failure.
5. `test_site_post` and `test_site_late` are nested subsets of `test_site_full`; they are robustness views, not independent replications.
6. This phase does not implement the maintenance knowledge graph, LLM grounding, or explainability evaluation required by the full project.